# Ablación J0–J3 (+ M-b3) — segmentación semántica de imágenes UAV oblicuas

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/datasantana/sam-image-processing/blob/main/notebooks/train_ablation.ipynb)

Este notebook **solo orquesta**: toda la lógica vive en el paquete `segpipe/` (probado con `pytest`).

| Paso | Qué cambia respecto al anterior |
|---|---|
| **J0** | Receta original: DeepLabv3+ R50, Resize 512, aumentos originales, Dice + CE ponderada, Adam 1e-4, batch 8, 50 épocas |
| **J1** | Recortes aleatorios 1024 a resolución nativa + inferencia por ventana deslizante |
| **J2** | Muestreo de recortes sesgado a clases raras |
| **J3** | Modelo → SegFormer mit-b3 |
| **M-b3** | Receta de Mauricio (aprox.), fuera de la escalera |

Siempre activo: clases desde `classes.json`, *background* (sin anotar) ignorado, split espacial por bloques con buffer, métricas a resolución nativa con IC por bootstrap de fotos.

**Antes de empezar:** *Entorno de ejecución → Cambiar tipo* → GPU (A100 o L4 recomendadas). Lo durable (split, checkpoints, logs, métricas) va a Google Drive; el dataset se descarga al disco local de la VM.

## 1. Código y dependencias

In [ ]:
# Clona (o actualiza) el repo en la VM. Si el repo es privado, usa un token de GitHub:
#   REPO_URL = "https://<token>@github.com/datasantana/sam-image-processing.git"
REPO_URL = "https://github.com/datasantana/sam-image-processing.git"
BRANCH = "main"
REPO_DIR = "/content/sam-image-processing"

import os, subprocess, sys
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=False)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

In [ ]:
# Usa el torch que trae Colab; instala solo lo que falte (nunca reinstala torch).
import importlib.util, subprocess, sys
need = {"segmentation_models_pytorch": "segmentation-models-pytorch", "transformers": "transformers",
        "albumentations": "albumentations", "timm": "timm", "yaml": "pyyaml", "cv2": "opencv-python-headless"}
missing = [pkg for mod, pkg in need.items() if importlib.util.find_spec(mod) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
from segpipe.env import gpu_info, versions
print("faltaban:", missing or "nada")
print(versions())
print(gpu_info())

## 2. Parámetros del estudio

`STUDY_DIR` debe estar en Google Drive: ahí quedan el split, los checkpoints por época, los logs y las métricas. Si la sesión se corta, **vuelve a ejecutar las mismas celdas**: cada corrida continúa desde su último `last.pt` y las terminadas (`DONE.json`) se saltan.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

STUDY_DIR = "/content/drive/MyDrive/segpipe/pascual_v1"   # durable (Drive)
DATASET_CFG = "configs/datasets/pascual.yaml"
PROTOCOL = "cv"          # "cv" = validación cruzada por bloques (artículo) | "fixed" = solo el pliegue 0
K_FOLDS, BLOCK_M, BUFFER_M, SPLIT_SEED = 5, 150.0, 50.0, 0
CONFIGS = ["J0", "J1", "J2", "J3", "M-b3"]
SEEDS = [42, 43, 44]

## 3. Dataset y split

`ensure_dataset` busca, en orden: copia ya validada en `/content/data` → descarga por file ID con tu cuenta (Drive API, archivo privado, reanudable) → Drive montado (`archive.drive_mounted_path` del YAML) → `SEGPACK_ZIP`/`SEGPACK_DIR`. Verifica tamaño y sha256, descomprime al disco local y valida contra `manifest.csv`.

In [ ]:
from segpipe.runner import prepare_study, write_ignored_report, describe_ladder
study = prepare_study(STUDY_DIR, DATASET_CFG, protocol=PROTOCOL, k=K_FOLDS, block_m=BLOCK_M,
                      buffer_m=BUFFER_M, split_seed=SPLIT_SEED)
print("clases del modelo:", study.schema.names, "| ignoradas:", study.schema.ignored)
write_ignored_report(study)   # % de píxeles sin anotar por foto → STUDY_DIR/dataset/
print()
print(describe_ladder(CONFIGS))

## 4. Corrida de humo (opcional, ~10–15 min)

Recorre descarga → validación → entrenamiento → checkpoint → **reanudación** → métricas con pocas fotos y 2 épocas, en un estudio aparte (`…_smoke`). La primera llamada se interrumpe a propósito tras la época 1; la segunda debe imprimir `[resume] … continúa en la época 2` y terminar con métricas. Ver `docs/smoke_run.md`.

In [ ]:
RUN_SMOKE = False
if RUN_SMOKE:
    from segpipe.runner import run_queue
    from segpipe.train import Interrupted
    smoke = prepare_study(STUDY_DIR + "_smoke", DATASET_CFG, protocol="fixed", k=K_FOLDS,
                          block_m=BLOCK_M, buffer_m=BUFFER_M, split_seed=SPLIT_SEED)
    kw = dict(seeds=[42], folds=[0], subset={"train": 6, "val": 2, "test": 3},
              overrides={"train": {"epochs": 2}, "data": {"samples_per_epoch": 32}, "eval": {"n_boot": 200}})
    for cfg in ["J1", "J3"]:
        try:
            run_queue(smoke, [cfg], stop_after_epochs=1, **kw)     # se interrumpe tras la época 1
        except Interrupted as e:
            print("→", e)
    res = run_queue(smoke, ["J1", "J3"], **kw)                    # reanuda y termina
    for r in res:
        print(r["run_dir"], "| test mIoU", round(r["test"]["mIoU"], 4), "| mejor época", r["best_epoch"])

## 5. Cola de corridas

Orden: semilla → pliegue → config (tras la primera semilla ya hay una CV completa de todas las configs). Usa `MAX_RUNS` para limitar cuántas corridas nuevas lanza esta sesión.

In [ ]:
from segpipe.runner import run_queue
run_queue(study, CONFIGS, SEEDS, dry_run=True)    # lista lo pendiente

In [ ]:
MAX_RUNS = None   # p. ej. 5 para sesiones cortas
results = run_queue(study, CONFIGS, SEEDS, max_runs=MAX_RUNS)

## 6. Tabla comparativa

Media ± DE entre semillas, IC 95 % por bootstrap de fotos (mismas fotos para todas las semillas), IoU por clase y diferencias pareadas J0→J1→J2→J3 (y cada config vs J0). Solo entran semillas con todos los pliegues terminados. Archivos en `STUDY_DIR/results/`.

In [ ]:
from IPython.display import Markdown, display
from segpipe import report
res = report.build(STUDY_DIR, CONFIGS, reference="J0")
display(Markdown(report.to_markdown(res)))

## 7. Curvas por época (opcional)

In [ ]:
import glob, pandas as pd, matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for cfg in CONFIGS:
    for p in sorted(glob.glob(f"{STUDY_DIR}/runs/{cfg}/seed{SEEDS[0]}/fold0/epoch_log.csv")):
        log = pd.read_csv(p)
        axes[0].plot(log.epoch, log.train_loss, label=cfg)
        axes[1].plot(log.epoch, log.val_mIoU, label=cfg)
        axes[2].plot(log.epoch, log.lr, label=cfg)
for ax, t in zip(axes, ["pérdida train", "mIoU val", "lr"]):
    ax.set_title(f"{t} (semilla {SEEDS[0]}, pliegue 0)"); ax.legend()
axes[2].set_yscale("log"); plt.show()